In [2]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def load_jsonl(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

df_aug = load_jsonl(DATA_DIR / 'aug_unaccented_reviews.jsonl')
df_shopee = load_jsonl(DATA_DIR / 'shopee_reviews_dataset.jsonl')

print(f"Kích thước tập Augmented: {df_aug.shape}")
print(f"Kích thước tập Shopee gốc: {df_shopee.shape}")

#(số lượng dòng, số lượng cột)

Kích thước tập Augmented: (1348, 4)
Kích thước tập Shopee gốc: (9599, 4)


In [4]:
print("Cột của tập Shopee:", df_shopee.columns.tolist())
print("Cột của tập Augmented:", df_aug.columns.tolist())

Cột của tập Shopee: ['id', 'review', 'rating', 'label']
Cột của tập Augmented: ['id', 'review', 'rating', 'label']


In [5]:
def check_data_integrity(df, dataset_name):
    print(f"\nĐánh giá dataset: {dataset_name}")
    print("1.Thông tin chung:")
    df.info()
    
    print("\n2.Missing values:")
    print(df.isnull().sum())
    
    print("\n3.Duplicate ID:")
    duplicate_ids = df['id'].duplicated().sum()
    print(f"Số ID bị trùng: {duplicate_ids}")
    
    print("\n4.Phân phối của nhãn:")
    print(df['label'].value_counts())

check_data_integrity(df_shopee, "Shopee Reviews")
check_data_integrity(df_aug, "Augmented Unaccented Reviews")

#677 + 671 = 1348 = số dòng => 0 có label khác ngoài positive và negative
#5965 + 3634 = 9599 = số dòng => 0 có label khác ngoài positive và negative


Đánh giá dataset: Shopee Reviews
1.Thông tin chung:
<class 'pandas.DataFrame'>
RangeIndex: 9599 entries, 0 to 9598
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      9599 non-null   str  
 1   review  9599 non-null   str  
 2   rating  9599 non-null   int64
 3   label   9599 non-null   str  
dtypes: int64(1), str(3)
memory usage: 300.1 KB

2.Missing values:
id        0
review    0
rating    0
label     0
dtype: int64

3.Duplicate ID:
Số ID bị trùng: 0

4.Phân phối của nhãn:
label
negative    5965
positive    3634
Name: count, dtype: int64

Đánh giá dataset: Augmented Unaccented Reviews
1.Thông tin chung:
<class 'pandas.DataFrame'>
RangeIndex: 1348 entries, 0 to 1347
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      1348 non-null   str  
 1   review  1348 non-null   str  
 2   rating  1348 non-null   int64
 3   label   1348 non-null   str  
dtypes: int64(1), 

In [6]:
print("tập Shopee gốc")
print(f"Rating Min: {df_shopee['rating'].min()}")
print(f"Rating Max: {df_shopee['rating'].max()}")

print("\ntập Augmented")
print(f"Rating Min: {df_aug['rating'].min()}")
print(f"Rating Max: {df_aug['rating'].max()}")

tập Shopee gốc
Rating Min: 1
Rating Max: 5

tập Augmented
Rating Min: 1
Rating Max: 5


In [7]:
print("Số lượng từng mức Rating tập Shopee gốc")
print(df_shopee['rating'].value_counts().sort_index())

print("\nSố lượng từng mức Rating tập Augmented")
print(df_aug['rating'].value_counts().sort_index())

Số lượng từng mức Rating tập Shopee gốc
rating
1    2972
2    1230
3    1782
4      96
5    3519
Name: count, dtype: int64

Số lượng từng mức Rating tập Augmented
rating
1    423
2    137
3    131
4     29
5    628
Name: count, dtype: int64


In [8]:
print("label theo Rating tập Shopee gốc")
print(pd.crosstab(df_shopee['rating'], df_shopee['label']))

print("\nlabel theo Rating tập Augmented")
print(pd.crosstab(df_aug['rating'], df_aug['label']))

label theo Rating tập Shopee gốc
label   negative  positive
rating                    
1           2958        14
2           1221         9
3           1691        91
4             29        67
5             66      3453

label theo Rating tập Augmented
label   negative  positive
rating                    
1            419         4
2            135         2
3            118        13
4              4        25
5              1       627


In [11]:
import emoji

def analyze_text_features(df, dataset_name):
    reviews = df['review'].astype(str)

    char_lengths = reviews.apply(len)
    word_lengths = reviews.apply(lambda x: len(x.split()))
    
    reviews_with_emoji = reviews.apply(lambda x: emoji.emoji_count(x) > 0).sum()
    
    print(f"dataset {dataset_name}")
    print(f"Review dài nhất : {char_lengths.max()} ký tự (tương đương ~{word_lengths.max()} từ)")
    print(f"Review ngắn nhất: {char_lengths.min()} ký tự (tương đương ~{word_lengths.min()} từ)")
    print(f"Số lượng review dùng Icon/Emoji: {reviews_with_emoji}")

analyze_text_features(df_shopee, "Shopee gốc")
analyze_text_features(df_aug, "Augmented")

dataset Shopee gốc
Review dài nhất : 2021 ký tự (tương đương ~444 từ)
Review ngắn nhất: 20 ký tự (tương đương ~3 từ)
Số lượng review dùng Icon/Emoji: 547
dataset Augmented
Review dài nhất : 2021 ký tự (tương đương ~444 từ)
Review ngắn nhất: 34 ký tự (tương đương ~6 từ)
Số lượng review dùng Icon/Emoji: 0
